# Feature Engineering - Demanda de Productos

## Objetivo

Este notebook crea features adicionales para el modelo de predicción de demanda de productos. Las features incluyen:

- Features temporales (día de semana cíclico, hora cíclica, mes, trimestre)
- Features de producto (categoría, precio, popularidad histórica)
- Features derivadas (lag features, rolling averages, tendencias)
- Features de festivos
- Normalización y escalado
- División train/validation/test

## Estructura

1. Carga de datos limpios
2. Features temporales
3. Features de producto
4. Features derivadas (lag, rolling averages)
5. Features de festivos
6. Codificación de variables categóricas
7. Normalización y escalado
8. División temporal
9. Validación de calidad

In [7]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from datetime import datetime, timedelta
import holidays
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Configurar rutas
DATA_DIR = Path('../data')

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 1. Carga de Datos Limpios

Cargamos el dataset limpio creado en el notebook de EDA, o lo recreamos si no existe.

In [8]:
# Intentar cargar dataset limpio, si no existe, recrearlo
try:
    df = pd.read_csv(DATA_DIR / 'datos_demanda_productos_limpio.csv')
    df['fecha'] = pd.to_datetime(df['fecha'])
    print("✅ Dataset limpio cargado desde archivo")
except FileNotFoundError:
    print("⚠️ Dataset limpio no encontrado, recreando desde datos originales...")
    # Cargar datos originales y procesar (código simplificado)
    orders_df = pd.read_csv(DATA_DIR / 'triplekb-orders-2026-03-08T15-37-31.csv')
    order_items_df = pd.read_csv(DATA_DIR / 'triplekb-order-items-2026-03-08T15-37-31.csv')
    
    orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
    order_items_df['order_date'] = pd.to_datetime(order_items_df['order_date'])
    
    demanda_df = order_items_df.merge(
        orders_df[['order_id', 'order_date', 'order_time', 'day_of_week', 'hour_of_day', 
                   'branch_id', 'delivery_zone_id']],
        on='order_id', how='left'
    )
    
    ve_holidays = holidays.Venezuela(years=[2025, 2026])
    demanda_df['es_festivo'] = demanda_df['order_date'].apply(
        lambda x: 'Sí' if x in ve_holidays else 'No'
    )
    
    df = demanda_df.groupby([
        'order_date', 'product_id', 'product_name', 'category_id', 'category_name',
        'day_of_week', 'hour_of_day', 'branch_id', 'delivery_zone_id', 'es_festivo'
    ]).agg({
        'quantity': 'sum',
        'product_price': 'mean'
    }).reset_index()
    
    df.columns = ['fecha', 'producto_id', 'producto_nombre', 'categoria_id', 'categoria_nombre',
                   'dia_semana', 'hora', 'sucursal_id', 'zona_entrega', 'es_festivo',
                   'cantidad_vendida', 'precio_unitario']
    df['promocion_activa'] = 'No'
    df['temperatura'] = np.nan
    print("✅ Dataset recreado")

print(f"\n📊 Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"📅 Rango de fechas: {df['fecha'].min().strftime('%Y-%m-%d')} a {df['fecha'].max().strftime('%Y-%m-%d')}")

✅ Dataset limpio cargado desde archivo

📊 Dimensiones: 33,126 filas × 14 columnas
📅 Rango de fechas: 2025-06-23 a 2026-03-08


## 2. Features Temporales

Creamos features temporales cíclicas y numéricas para capturar patrones temporales.

In [9]:
# Features temporales básicas
df['mes'] = df['fecha'].dt.month
df['trimestre'] = df['fecha'].dt.quarter
df['dia_del_mes'] = df['fecha'].dt.day
df['semana_del_año'] = df['fecha'].dt.isocalendar().week
df['año'] = df['fecha'].dt.year

# Días desde inicio del período
df['dias_desde_inicio'] = (df['fecha'] - df['fecha'].min()).dt.days

# Features cíclicas para día de semana (sin y cos para capturar periodicidad)
df['dia_semana_num'] = df['dia_semana'].map({
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
    'Friday': 4, 'Saturday': 5, 'Sunday': 6
})
df['dia_semana_sin'] = np.sin(2 * np.pi * df['dia_semana_num'] / 7)
df['dia_semana_cos'] = np.cos(2 * np.pi * df['dia_semana_num'] / 7)

# Features cíclicas para hora del día
df['hora_sin'] = np.sin(2 * np.pi * df['hora'] / 24)
df['hora_cos'] = np.cos(2 * np.pi * df['hora'] / 24)

# Features cíclicas para mes
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)

print("✅ Features temporales creadas")
print(f"\n📊 Features temporales agregadas:")
print(f"   - mes, trimestre, dia_del_mes, semana_del_año")
print(f"   - dias_desde_inicio")
print(f"   - dia_semana_sin, dia_semana_cos (cíclicas)")
print(f"   - hora_sin, hora_cos (cíclicas)")
print(f"   - mes_sin, mes_cos (cíclicas)")

✅ Features temporales creadas

📊 Features temporales agregadas:
   - mes, trimestre, dia_del_mes, semana_del_año
   - dias_desde_inicio
   - dia_semana_sin, dia_semana_cos (cíclicas)
   - hora_sin, hora_cos (cíclicas)
   - mes_sin, mes_cos (cíclicas)


## 3. Features de Producto

Creamos features relacionadas con los productos y su popularidad histórica.

In [10]:
# Popularidad histórica del producto (promedio de ventas en los últimos 7, 14, 30 días)
df = df.sort_values(['producto_id', 'fecha'])

# Rolling averages por producto
for window in [7, 14, 30]:
    df[f'ventas_promedio_{window}d'] = df.groupby('producto_id')['cantidad_vendida'].transform(
        lambda x: x.shift(1).rolling(window=window, min_periods=1).mean()
    )

# Precio del producto (ya existe como precio_unitario)
# Agregar precio relativo (comparado con promedio de la categoría)
precio_promedio_categoria = df.groupby('categoria_nombre')['precio_unitario'].transform('mean')
df['precio_relativo_categoria'] = df['precio_unitario'] / precio_promedio_categoria

print("✅ Features de producto creadas")
print(f"\n📊 Features de producto agregadas:")
print(f"   - ventas_promedio_7d, ventas_promedio_14d, ventas_promedio_30d")
print(f"   - precio_relativo_categoria")

✅ Features de producto creadas

📊 Features de producto agregadas:
   - ventas_promedio_7d, ventas_promedio_14d, ventas_promedio_30d
   - precio_relativo_categoria


## 4. Features Derivadas (Lag Features y Tendencias)

Creamos features que capturan patrones históricos y tendencias.

In [11]:
# Lag features: ventas de días anteriores (1, 7, 14 días)
for lag in [1, 7, 14]:
    df[f'lag_{lag}d'] = df.groupby('producto_id')['cantidad_vendida'].shift(lag)

# Tendencias: diferencia entre ventas recientes y pasadas
df['tendencia_7d'] = df.groupby('producto_id')['cantidad_vendida'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).mean() - 
              x.shift(8).rolling(window=7, min_periods=1).mean()
)

# Desviación estándar de ventas recientes (volatilidad)
df['volatilidad_7d'] = df.groupby('producto_id')['cantidad_vendida'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).std()
)

# Máximo y mínimo de ventas recientes
df['max_ventas_7d'] = df.groupby('producto_id')['cantidad_vendida'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).max()
)
df['min_ventas_7d'] = df.groupby('producto_id')['cantidad_vendida'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).min()
)

print("✅ Features derivadas creadas")
print(f"\n📊 Features derivadas agregadas:")
print(f"   - lag_1d, lag_7d, lag_14d")
print(f"   - tendencia_7d")
print(f"   - volatilidad_7d")
print(f"   - max_ventas_7d, min_ventas_7d")

✅ Features derivadas creadas

📊 Features derivadas agregadas:
   - lag_1d, lag_7d, lag_14d
   - tendencia_7d
   - volatilidad_7d
   - max_ventas_7d, min_ventas_7d


## 5. Features de Festivos

Ya tenemos la información de festivos, pero podemos crear features adicionales.

In [12]:
# Codificar festivos como binario
df['es_festivo_num'] = df['es_festivo'].map({'Sí': 1, 'No': 0})

# Días hasta el próximo festivo
ve_holidays = holidays.Venezuela(years=[2025, 2026])
# Convertir Timestamp a date para comparación
fecha_min = df['fecha'].min().date()
fechas_festivos = sorted([fecha for fecha in ve_holidays.keys() if fecha >= fecha_min])

def dias_hasta_proximo_festivo(fecha):
    # Convertir Timestamp a date si es necesario
    fecha_date = fecha.date() if isinstance(fecha, pd.Timestamp) else fecha
    for festivo in fechas_festivos:
        if festivo >= fecha_date:
            return (festivo - fecha_date).days
    return np.nan

df['dias_hasta_proximo_festivo'] = df['fecha'].apply(dias_hasta_proximo_festivo)

# Días desde el último festivo
def dias_desde_ultimo_festivo(fecha):
    # Convertir Timestamp a date si es necesario
    fecha_date = fecha.date() if isinstance(fecha, pd.Timestamp) else fecha
    for festivo in reversed(fechas_festivos):
        if festivo <= fecha_date:
            return (fecha_date - festivo).days
    return np.nan

df['dias_desde_ultimo_festivo'] = df['fecha'].apply(dias_desde_ultimo_festivo)

print("✅ Features de festivos creadas")
print(f"\n📊 Features de festivos agregadas:")
print(f"   - es_festivo_num")
print(f"   - dias_hasta_proximo_festivo")
print(f"   - dias_desde_ultimo_festivo")

✅ Features de festivos creadas

📊 Features de festivos agregadas:
   - es_festivo_num
   - dias_hasta_proximo_festivo
   - dias_desde_ultimo_festivo


## 6. Codificación de Variables Categóricas

Codificamos variables categóricas para que puedan ser usadas en los modelos.

In [13]:
# Label encoding para categorías con muchos valores únicos
le_categoria = LabelEncoder()
df['categoria_encoded'] = le_categoria.fit_transform(df['categoria_nombre'])

le_producto = LabelEncoder()
df['producto_encoded'] = le_producto.fit_transform(df['producto_id'])

# One-hot encoding para variables con pocos valores únicos
df['es_fin_semana'] = df['dia_semana'].apply(lambda x: 1 if x in ['Saturday', 'Sunday'] else 0)
df['promocion_activa_num'] = df['promocion_activa'].map({'Sí': 1, 'No': 0})

print("✅ Variables categóricas codificadas")
print(f"\n📊 Codificaciones realizadas:")
print(f"   - categoria_encoded (Label Encoding)")
print(f"   - producto_encoded (Label Encoding)")
print(f"   - es_fin_semana (binario)")
print(f"   - promocion_activa_num (binario)")

✅ Variables categóricas codificadas

📊 Codificaciones realizadas:
   - categoria_encoded (Label Encoding)
   - producto_encoded (Label Encoding)
   - es_fin_semana (binario)
   - promocion_activa_num (binario)


## 7. Normalización y Escalado

Normalizamos las features numéricas para mejorar el rendimiento de los modelos.

In [14]:
# Seleccionar features numéricas para normalizar
features_numericas = [
    'precio_unitario', 'precio_relativo_categoria',
    'ventas_promedio_7d', 'ventas_promedio_14d', 'ventas_promedio_30d',
    'lag_1d', 'lag_7d', 'lag_14d',
    'tendencia_7d', 'volatilidad_7d',
    'max_ventas_7d', 'min_ventas_7d',
    'dias_desde_inicio', 'dias_hasta_proximo_festivo', 'dias_desde_ultimo_festivo'
]

# Rellenar valores NaN con 0 o medianas antes de normalizar
for feature in features_numericas:
    if feature in df.columns:
        df[feature] = df[feature].fillna(df[feature].median() if df[feature].notna().sum() > 0 else 0)

# Normalizar features (se hará después de dividir train/test para evitar data leakage)
print("✅ Features numéricas preparadas para normalización")
print(f"\n📊 Features a normalizar: {len(features_numericas)}")
print(f"   {', '.join(features_numericas[:5])}...")

✅ Features numéricas preparadas para normalización

📊 Features a normalizar: 15
   precio_unitario, precio_relativo_categoria, ventas_promedio_7d, ventas_promedio_14d, ventas_promedio_30d...


## 8. División Temporal Train/Validation/Test

Dividimos los datos temporalmente para evitar data leakage.

In [15]:
# Ordenar por fecha
df = df.sort_values('fecha').reset_index(drop=True)

# División temporal según el plan:
# Train: Junio 2025 - Diciembre 2025
# Validation: Enero 2026 - Febrero 2026
# Test: Marzo 2026

train_end = pd.Timestamp('2025-12-31')
val_end = pd.Timestamp('2026-02-28')

train_df = df[df['fecha'] <= train_end].copy()
val_df = df[(df['fecha'] > train_end) & (df['fecha'] <= val_end)].copy()
test_df = df[df['fecha'] > val_end].copy()

print("=" * 80)
print("DIVISIÓN TEMPORAL")
print("=" * 80)
print(f"\n📊 Train:")
print(f"   Fechas: {train_df['fecha'].min().strftime('%Y-%m-%d')} a {train_df['fecha'].max().strftime('%Y-%m-%d')}")
print(f"   Registros: {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")

print(f"\n📊 Validation:")
print(f"   Fechas: {val_df['fecha'].min().strftime('%Y-%m-%d')} a {val_df['fecha'].max().strftime('%Y-%m-%d')}")
print(f"   Registros: {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")

print(f"\n📊 Test:")
print(f"   Fechas: {test_df['fecha'].min().strftime('%Y-%m-%d')} a {test_df['fecha'].max().strftime('%Y-%m-%d')}")
print(f"   Registros: {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")

DIVISIÓN TEMPORAL

📊 Train:
   Fechas: 2025-06-23 a 2025-12-31
   Registros: 22,153 (66.9%)

📊 Validation:
   Fechas: 2026-01-01 a 2026-02-28
   Registros: 9,614 (29.0%)

📊 Test:
   Fechas: 2026-03-01 a 2026-03-08
   Registros: 1,359 (4.1%)


## 9. Normalización de Features (después de división)

Aplicamos normalización solo a los datos de entrenamiento y luego transformamos validation y test.

In [16]:
# Seleccionar features numéricas disponibles
features_numericas_disponibles = [f for f in features_numericas if f in df.columns]

# Aplicar normalización
scaler = StandardScaler()

# Entrenar scaler solo con datos de train
train_df_scaled = train_df.copy()
val_df_scaled = val_df.copy()
test_df_scaled = test_df.copy()

train_df_scaled[features_numericas_disponibles] = scaler.fit_transform(
    train_df[features_numericas_disponibles]
)
val_df_scaled[features_numericas_disponibles] = scaler.transform(
    val_df[features_numericas_disponibles]
)
test_df_scaled[features_numericas_disponibles] = scaler.transform(
    test_df[features_numericas_disponibles]
)

print("✅ Features normalizadas")
print(f"\n📊 Features normalizadas: {len(features_numericas_disponibles)}")

✅ Features normalizadas

📊 Features normalizadas: 15


## 10. Selección de Features Finales

Seleccionamos las features que se usarán en el modelo.

In [17]:
# Seleccionar features finales para el modelo
feature_columns = [
    # Temporales
    'mes', 'trimestre', 'semana_del_año', 'dias_desde_inicio',
    'dia_semana_sin', 'dia_semana_cos', 'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos',
    # Producto
    'precio_unitario', 'precio_relativo_categoria',
    'ventas_promedio_7d', 'ventas_promedio_14d', 'ventas_promedio_30d',
    'categoria_encoded', 'producto_encoded',
    # Derivadas
    'lag_1d', 'lag_7d', 'lag_14d',
    'tendencia_7d', 'volatilidad_7d', 'max_ventas_7d', 'min_ventas_7d',
    # Festivos
    'es_festivo_num', 'es_fin_semana', 'dias_hasta_proximo_festivo', 'dias_desde_ultimo_festivo',
    # Otros
    'promocion_activa_num'
]

# Filtrar features que existen
feature_columns = [f for f in feature_columns if f in train_df_scaled.columns]

# Crear datasets finales
X_train = train_df_scaled[feature_columns].copy()
y_train = train_df_scaled['cantidad_vendida'].copy()

X_val = val_df_scaled[feature_columns].copy()
y_val = val_df_scaled['cantidad_vendida'].copy()

X_test = test_df_scaled[feature_columns].copy()
y_test = test_df_scaled['cantidad_vendida'].copy()

print("=" * 80)
print("DATASETS FINALES PARA MODELADO")
print("=" * 80)
print(f"\n📊 Features seleccionadas: {len(feature_columns)}")
print(f"\n📋 Features:")
for i, feat in enumerate(feature_columns, 1):
    print(f"   {i:2d}. {feat}")

print(f"\n📊 Dimensiones:")
print(f"   Train: X={X_train.shape}, y={y_train.shape}")
print(f"   Validation: X={X_val.shape}, y={y_val.shape}")
print(f"   Test: X={X_test.shape}, y={y_test.shape}")

# Guardar datasets
X_train.to_csv(DATA_DIR / 'X_train_demanda.csv', index=False)
y_train.to_csv(DATA_DIR / 'y_train_demanda.csv', index=False)
X_val.to_csv(DATA_DIR / 'X_val_demanda.csv', index=False)
y_val.to_csv(DATA_DIR / 'y_val_demanda.csv', index=False)
X_test.to_csv(DATA_DIR / 'X_test_demanda.csv', index=False)
y_test.to_csv(DATA_DIR / 'y_test_demanda.csv', index=False)

print(f"\n✅ Datasets guardados en {DATA_DIR}")

DATASETS FINALES PARA MODELADO

📊 Features seleccionadas: 29

📋 Features:
    1. mes
    2. trimestre
    3. semana_del_año
    4. dias_desde_inicio
    5. dia_semana_sin
    6. dia_semana_cos
    7. hora_sin
    8. hora_cos
    9. mes_sin
   10. mes_cos
   11. precio_unitario
   12. precio_relativo_categoria
   13. ventas_promedio_7d
   14. ventas_promedio_14d
   15. ventas_promedio_30d
   16. categoria_encoded
   17. producto_encoded
   18. lag_1d
   19. lag_7d
   20. lag_14d
   21. tendencia_7d
   22. volatilidad_7d
   23. max_ventas_7d
   24. min_ventas_7d
   25. es_festivo_num
   26. es_fin_semana
   27. dias_hasta_proximo_festivo
   28. dias_desde_ultimo_festivo
   29. promocion_activa_num

📊 Dimensiones:
   Train: X=(22153, 29), y=(22153,)
   Validation: X=(9614, 29), y=(9614,)
   Test: X=(1359, 29), y=(1359,)

✅ Datasets guardados en ../data


## 11. Validación de Calidad de Datos

Verificamos que no haya problemas con los datos preparados.

In [18]:
print("=" * 80)
print("VALIDACIÓN DE CALIDAD DE DATOS")
print("=" * 80)

# Verificar valores faltantes
print(f"\n📊 Valores faltantes en Train:")
missing_train = X_train.isnull().sum()
if missing_train.sum() > 0:
    print(missing_train[missing_train > 0])
else:
    print("   ✅ No hay valores faltantes")

print(f"\n📊 Valores faltantes en Validation:")
missing_val = X_val.isnull().sum()
if missing_val.sum() > 0:
    print(missing_val[missing_val > 0])
else:
    print("   ✅ No hay valores faltantes")

print(f"\n📊 Valores faltantes en Test:")
missing_test = X_test.isnull().sum()
if missing_test.sum() > 0:
    print(missing_test[missing_test > 0])
else:
    print("   ✅ No hay valores faltantes")

# Verificar valores infinitos
print(f"\n📊 Valores infinitos:")
print(f"   Train: {np.isinf(X_train.select_dtypes(include=[np.number])).sum().sum()}")
print(f"   Validation: {np.isinf(X_val.select_dtypes(include=[np.number])).sum().sum()}")
print(f"   Test: {np.isinf(X_test.select_dtypes(include=[np.number])).sum().sum()}")

# Estadísticas básicas de la variable objetivo
print(f"\n📊 Estadísticas de cantidad_vendida (y):")
print(f"   Train - Media: {y_train.mean():.2f}, Mediana: {y_train.median():.2f}, Std: {y_train.std():.2f}")
print(f"   Validation - Media: {y_val.mean():.2f}, Mediana: {y_val.median():.2f}, Std: {y_val.std():.2f}")
print(f"   Test - Media: {y_test.mean():.2f}, Mediana: {y_test.median():.2f}, Std: {y_test.std():.2f}")

print(f"\n✅ Validación completada")

VALIDACIÓN DE CALIDAD DE DATOS

📊 Valores faltantes en Train:
   ✅ No hay valores faltantes

📊 Valores faltantes en Validation:
   ✅ No hay valores faltantes

📊 Valores faltantes en Test:
   ✅ No hay valores faltantes

📊 Valores infinitos:
   Train: 0
   Validation: 0
   Test: 0

📊 Estadísticas de cantidad_vendida (y):
   Train - Media: 1.76, Mediana: 1.00, Std: 1.60
   Validation - Media: 1.75, Mediana: 1.00, Std: 1.73
   Test - Media: 1.72, Mediana: 1.00, Std: 1.34

✅ Validación completada


## Resumen

Este notebook ha preparado los datos para el modelado de demanda de productos:

1. ✅ Features temporales creadas (cíclicas y numéricas)
2. ✅ Features de producto creadas (popularidad, precio relativo)
3. ✅ Features derivadas creadas (lag, rolling averages, tendencias)
4. ✅ Features de festivos creadas
5. ✅ Variables categóricas codificadas
6. ✅ Features normalizadas
7. ✅ División temporal realizada (train/validation/test)
8. ✅ Datasets guardados para modelado

### Próximos Pasos

El siguiente notebook (`05_modelo_demanda_productos.ipynb`) utilizará estos datasets preparados para entrenar modelos de predicción de demanda.